# Semana 1 — Exploração do Workspace Databricks

**Objetivo:** Familiarizar com o ambiente Databricks usando `dbutils`, `spark` e magic commands.

> ⚠️ **Este notebook foi desenvolvido e validado com Serverless compute.**
> Algumas APIs legadas não estão disponíveis no Serverless — as células abaixo já refletem as adaptações necessárias:
> - `sc` (SparkContext) **não existe** no Serverless → use `spark`
> - `delta.`caminho`` em arquivos CSV → use `createOrReplaceTempView` + `%sql`
> - Colunas com espaços no nome → use backticks no SQL
> - `dbfs:/tmp/` está **desabilitado** neste workspace (DBFS root bloqueado por Unity Catalog) → use **Unity Catalog Volumes** (`/Volumes/workspace/estudos/...`)

---
## Exercício — Exploração do Workspace

Execute as células abaixo em sequência em um notebook Databricks com Serverless conectado.

In [0]:
# Célula 1: Explorar a raiz do DBFS
# dbutils.fs é a API de filesystem do Databricks — acessa DBFS, Volumes e caminhos de cloud
# display() renderiza tabelas ricas no notebook (não disponível fora do Databricks)
display(dbutils.fs.ls("dbfs:/"))

In [0]:
# Célula 2: Datasets de exemplo pré-instalados no Databricks
# O Databricks inclui datasets públicos em dbfs:/databricks-datasets/ para aprendizado
display(dbutils.fs.ls("dbfs:/databricks-datasets/"))

In [0]:
# Célula 3: Runtime e versão do Spark
# IMPORTANTE: sc (SparkContext) NÃO existe no Serverless → use sempre spark (SparkSession)
# sc.version lança NotImplementedError no Serverless
import sys

print(f"Spark version  : {spark.version}")
print(f"Python version : {sys.version}")
print(f"Compute type   : Serverless (sc/SparkContext não disponível)")

In [0]:
# Célula 4: Ler CSV e registrar como view temporária
# IMPORTANTE: este dataset é CSV puro — NÃO use delta.`caminho` (lança DELTA_MISSING_TRANSACTION_LOG)
# createOrReplaceTempView() permite referenciar o DataFrame em células %sql
df = spark.read.csv(
    "dbfs:/databricks-datasets/samples/population-vs-price/data_geo.csv",
    header=True,
    inferSchema=True
)
df.printSchema()
display(df)

# Registrar como view para a célula %sql abaixo
df.createOrReplaceTempView("population_price")

In [0]:
%sql
-- Célula 5: Top 10 estados por preço médio de venda (2015)
-- IMPORTANTE: colunas com espaços precisam de backticks — `2015 median sales price`
-- Requer que a view `population_price` tenha sido criada na célula anterior
SELECT `State Code`, State, `2015 median sales price`
FROM (
  SELECT *,
         ROW_NUMBER() OVER (ORDER BY `2015 median sales price` DESC) AS rank
  FROM population_price
)
WHERE rank <= 10
ORDER BY rank

---
## Q1 — dbutils.fs profundo

Usando apenas `dbutils.fs` (sem `spark.read`), execute as 6 tarefas abaixo:

1. Listar `dbfs:/databricks-datasets/`
2. Identificar 3 datasets com suas descrições
3. Criar a estrutura de pastas (usando Unity Catalog Volume — recomendado no Serverless)
4. Criar um arquivo com `dbutils.fs.put()`
5. Ler o arquivo com `dbutils.fs.head()`
6. Deletar com `dbutils.fs.rm(..., recurse=True)`

> 💡 **Por que Unity Catalog Volume em vez de `dbfs:/tmp/`?**
> No Serverless, `dbfs:/tmp/` funciona mas é uma área compartilhada e não gerenciada.
> **UC Volumes** (`/Volumes/catalog/schema/volume/`) são a forma moderna e recomendada:
> persistentes, versionados pelo Unity Catalog, com controle de acesso granular.

In [0]:
# Item 1: Listar dbfs:/databricks-datasets/
datasets = dbutils.fs.ls("dbfs:/databricks-datasets/")
for item in datasets:
    print(f"{item.name:<40} {'(diretório)' if item.size == 0 else f'{item.size:,} bytes'}")

In [0]:
# Item 2: Identificar 3 datasets com descrições
# Lê o arquivo README de cada dataset para entender o conteúdo
datasets_selecionados = [
    "airlines",
    "sample_logs",
    "bikeSharing",
]

for nome in datasets_selecionados:
    path = f"dbfs:/databricks-datasets/{nome}/"
    try:
        readme_files = [f for f in dbutils.fs.ls(path) if "README" in f.name.upper() or "readme" in f.name]
        if readme_files:
            conteudo = dbutils.fs.head(readme_files[0].path, 300)
            print(f"\n{'='*60}")
            print(f"Dataset: {nome}")
            print(f"{'='*60}")
            print(conteudo[:300])
        else:
            arquivos = dbutils.fs.ls(path)
            print(f"\nDataset: {nome} — {len(arquivos)} arquivos, sem README")
    except Exception as e:
        print(f"Dataset {nome}: {e}")

In [0]:
%sql
-- # Item 3a: Verificar os catálogos disponíveis no Unity Catalog
-- # O catálogo 'workspace' é o padrão do trial — é onde criaremos nosso schema e volume
SHOW CATALOGS

In [0]:
%sql
-- Item 3b: Criar schema e volume no Unity Catalog
-- Volumes são a alternativa moderna ao dbfs:/tmp/ — gerenciados, com controle de acesso
-- Caminho de acesso via dbutils: /Volumes/<catalog>/<schema>/<volume>/
CREATE SCHEMA IF NOT EXISTS workspace.estudos
  COMMENT 'Schema para exercícios do curso Semana 1';

CREATE VOLUME IF NOT EXISTS workspace.estudos.meu_volume
  COMMENT 'Volume gerenciado para exercícios de dbutils.fs';

In [0]:
# Item 3c: Definir o caminho base no Volume criado
# Formato: /Volumes/<catalog>/<schema>/<volume>/<pasta>
base_path = "/Volumes/workspace/estudos/meu_volume/exercicio_q1"
print(f"Caminho base: {base_path}")

In [0]:
# Item 3d: Criar a estrutura de pastas no Volume
# mkdirs cria o diretório e todos os intermediários (equivalente a mkdir -p)
dbutils.fs.mkdirs(base_path)
print(f"Diretório criado: {base_path}")

In [0]:
# Item 4: Criar arquivo de texto com dbutils.fs.put()
# overwrite=True substitui o arquivo se já existir
arquivo_path = f"{base_path}/notas.txt"
conteudo = """Aprendizados - Semana 1
========================
- dbutils.fs.ls()    : lista arquivos/diretórios
- dbutils.fs.mkdirs(): cria diretórios recursivamente
- dbutils.fs.put()   : escreve arquivo de texto
- dbutils.fs.head()  : lê início do arquivo (bytes)
- dbutils.fs.rm()    : remove arquivo ou diretório
- Unity Catalog Volumes > dbfs:/tmp/ no Serverless
"""

dbutils.fs.put(arquivo_path, conteudo, overwrite=True)
print(f"Arquivo gravado: {arquivo_path}")

In [0]:
# Item 5: Ler o arquivo com dbutils.fs.head()
# head() retorna os primeiros N bytes como string — útil para inspecionar arquivos grandes
conteudo_lido = dbutils.fs.head(arquivo_path, 500)
print(f"Conteúdo de {arquivo_path}:\n")
print(conteudo_lido)

In [0]:
# Verificar o conteúdo do diretório antes de deletar
print("Conteúdo do diretório:")
arquivos = dbutils.fs.ls(base_path)
for f in arquivos:
    print(f"  {f.name} ({f.size} bytes)")

In [0]:
# Item 6: Deletar o diretório com recurse=True (apaga tudo dentro)
dbutils.fs.rm(base_path, recurse=True)

# Verificar que foi deletado (deve lançar exceção ao tentar listar)
try:
    dbutils.fs.ls(base_path)
    print("AVISO: diretório ainda existe")
except Exception:
    print(f"Confirmado: {base_path} foi removido com sucesso")

### Reflexão Q1

**Pergunta:** O que acontece se você fechar o notebook e reabrir — os arquivos criados ainda existem?

**Resposta:** **Sim, os arquivos persistem.** Diferente do ciclo de vida de um cluster onde o driver pode ser reiniciado e variáveis Python se perdem, arquivos gravados em `dbfs:/` ou em **Unity Catalog Volumes** são armazenamento persistente. Eles só desaparecem quando explicitamente deletados (`fs.rm`). Isso vale tanto para o Serverless quanto para clusters Classic.

---
## Q2 — Magic commands

Magic commands são atalhos de linguagem que mudam o contexto de execução de uma célula.
Em notebooks Databricks, os 5 principais são:

| Magic    | Linguagem/Ação       | Exemplo                                    |
|----------|---------------------|--------------------------------------------|
| `%md`    | Markdown            | `%md # Título` → renderiza HTML            |
| `%fs`    | dbutils.fs         | `%fs ls /databricks-datasets/`             |
| `%sql`   | SQL                 | `%sql SELECT current_catalog()`            |
| `%sh`    | Shell (bash)        | `%sh pip list` — roda no driver node       |
| `%run`   | Executa outro notebook | `%run ./outro_notebook`               |

> ⚠️ `%run` compartilha o estado do Python: variáveis e funções definidas no notebook chamado ficam disponíveis na sessão atual.

# Magic Command: %md

Esta célula usa `%md` para renderizar **Markdown** diretamente no notebook.

Você pode usar:
- **Negrito**, *itálico*, `código inline`
- Tabelas, listas, títulos com `#`, `##`, `###`
- Blocos de código com ` ``` `
- Links: [Databricks Docs](https://docs.databricks.com)

> 💡 Use `%md` para documentar o raciocínio entre células de código — é o que torna um notebook legível como guia de estudo.

In [0]:
%fs
ls /databricks-datasets/

In [0]:
%sql
-- Magic %sql: executa SQL diretamente — sem precisar de spark.sql() ou createOrReplaceTempView()
-- Útil para exploração rápida, DDL e queries ad-hoc
SELECT current_catalog(), current_schema(), current_date()

In [0]:
%sh
# Magic %sh: executa shell bash no driver node do cluster (ou nó Serverless)
# Variáveis Python do notebook NÃO estão disponíveis aqui — é um processo separado
# Útil para: pip list, verificar sistema de arquivos local, curl, etc.
echo "=== Python packages instalados ==="
pip list | grep -E "databricks|spark|mlflow|pandas"
echo ""
echo "=== Java version ==="
java -version 2>&1

In [0]:
# Contexto do notebook atual via dbutils.notebook
# Útil para construir caminhos relativos ao notebook corrente (ex: para %run)
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
print(f"Caminho deste notebook: {notebook_path}")
print(f"Diretório pai: {'/'.join(notebook_path.split('/')[:-1])}")

In [0]:
%run ./q2_helper


In [0]:
# Usar variável e função definidas no q2_helper via %run
# Prova que o estado foi compartilhado entre os dois notebooks
print(f"Variável do helper: {mensagem_helper}")
print(f"Função do helper  : {saudacao_q2('Estudante')}")

---
## Resumo — Semana 1

### O que praticamos

| Tópico | API / Comando | Nota |
|--------|--------------|------|
| Filesystem | `dbutils.fs.ls/put/head/rm` | Preferir UC Volumes a `dbfs:/tmp/` |
| Spark session | `spark.version`, `spark.read.csv()` | `sc` não existe no Serverless |
| Views temporárias | `createOrReplaceTempView()` | Alternativa a `delta.path` em CSV |
| SQL em notebooks | `%sql` magic | Backticks para colunas com espaço |
| Markdown | `%md` magic | Documenta raciocínio no notebook |
| Shell | `%sh` magic | Roda no driver node — isolado do Python |
| Execução encadeada | `%run ./outro_notebook` | Compartilha variáveis e funções |
| Unity Catalog | `CREATE SCHEMA/VOLUME` | Caminho: `/Volumes/<cat>/<schema>/<vol>/` |

### Reflexão Q2: onde roda o `%sh`?
O `%sh` executa um processo bash no **driver node** do cluster (ou nó Serverless).
Ele é **completamente isolado** do interpretador Python do notebook — variáveis Python não são visíveis.
É útil para operações de sistema: instalar pacotes com `pip`, verificar arquivos locais, testar conectividade.